In [ ]:
# ============================================================
# ASR Extraction Pipeline
# Models : Whisper-large-v3 + MMS-1B
# Dataset: aliyzd95/common_voice_22_0_fa
# ============================================================

import os
import json
import time
import pandas as pd
import torch
from datasets import load_dataset, Audio
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    AutoProcessor,
    AutoModelForCTC
)

# ============================================================
# Config
# ============================================================
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_FP16 = torch.cuda.is_available()

LOCAL_OUTPUT_CSV  = "/kaggle/working/asr_outputs_raw.csv"
LOCAL_OUTPUT_XLSX = "/kaggle/working/asr_outputs_raw.xlsx"
LOCAL_STATE_JSON  = "/kaggle/working/asr_state.json"

CHECKPOINT_INTERVAL = 100
BATCH_SIZE = 8          # tune for your GPU: try 4, 8, 12 (lower if OOM)
DISPLAY_EVERY = 20      # print progress every 20 samples

# ============================================================
# Helpers
# ============================================================
def report(msg):
    print(msg, flush=True)


def save_state(extra=None):
    state = {
        "last_run": time.strftime("%Y-%m-%d %H:%M:%S"),
        "device": DEVICE,
        "batch_size": BATCH_SIZE,
    }
    if extra:
        state.update(extra)
    with open(LOCAL_STATE_JSON, "w") as f:
        json.dump(state, f, indent=2)


# ============================================================
# Load dataset
# ============================================================
report("Loading dataset...")
dataset = load_dataset("aliyzd95/common_voice_22_0_fa")
dataset = dataset.cast_column("audio", Audio(sampling_rate=16000))

for split_name in dataset.keys():
    report(f"  split '{split_name}': {len(dataset[split_name])} samples")

# ============================================================
# Load models
# ============================================================
report("Loading Whisper-large-v3...")
whisper_processor = WhisperProcessor.from_pretrained("openai/whisper-large-v3")
whisper_model = WhisperForConditionalGeneration.from_pretrained(
    "openai/whisper-large-v3"
).to(DEVICE)
if USE_FP16:
    whisper_model = whisper_model.half()
whisper_model.eval()
whisper_model.generation_config.language = "fa"
whisper_model.generation_config.task = "transcribe"

report("Loading MMS-1B...")
mms_processor = AutoProcessor.from_pretrained("facebook/mms-1b-all")
mms_processor.tokenizer.set_target_lang("fas")
mms_model = AutoModelForCTC.from_pretrained(
    "facebook/mms-1b-all",
    target_lang="fas"
).to(DEVICE)
if USE_FP16:
    mms_model = mms_model.half()
mms_model.eval()

# ============================================================
# Inference functions
# ============================================================
@torch.no_grad()
def whisper_batch_transcribe(audios):
    inputs = whisper_processor(
        audios, sampling_rate=16000, return_tensors="pt"
    )
    input_features = inputs.input_features.to(DEVICE)
    if USE_FP16:
        input_features = input_features.half()

    pred_ids = whisper_model.generate(
        input_features,
        language="fa",
        task="transcribe"
    )
    return whisper_processor.batch_decode(pred_ids, skip_special_tokens=True)


@torch.no_grad()
def mms_batch_transcribe(audios):
    inputs = mms_processor(
        audios, sampling_rate=16000, padding=True, return_tensors="pt"
    )
    input_values = inputs.input_values.to(DEVICE)
    if USE_FP16:
        input_values = input_values.half()

    logits = mms_model(input_values).logits
    pred_ids = torch.argmax(logits, dim=-1)
    return mms_processor.batch_decode(pred_ids)


# ============================================================
# Checkpoint handling
# ============================================================
def load_checkpoint_df():
    if os.path.exists(LOCAL_OUTPUT_CSV):
        df = pd.read_csv(LOCAL_OUTPUT_CSV)
        if "id" not in df.columns:
            raise ValueError("CSV exists but 'id' column is missing")
        processed = set(df["id"].astype(str))
        return df, processed
    return (
        pd.DataFrame(columns=["split", "id", "ground_truth", "whisper", "mms"]),
        set(),
    )


def save_checkpoint_df(df):
    df.to_csv(LOCAL_OUTPUT_CSV, index=False)
    try:
        df.to_excel(LOCAL_OUTPUT_XLSX, index=False)
    except Exception as exc:
        report(f"  [warn] XLSX save failed: {exc}")
    report(f"  [checkpoint] saved {len(df)} rows")


# ============================================================
# Main split processor
# ============================================================
def process_split(split_name, df, processed_ids):
    split_data = dataset[split_name]
    total = len(split_data)

    new_rows = []
    pending_items = []
    processed_in_split = 0

    for idx in range(total):
        item = split_data[idx]
        sample_id = str(item["path"])

        if sample_id in processed_ids:
            continue

        pending_items.append(item)

        if len(pending_items) < BATCH_SIZE:
            continue

        # ---- process one batch ----
        audios    = [x["audio"]["array"] for x in pending_items]
        gt_texts  = [x["sentence"] for x in pending_items]
        ids       = [str(x["path"]) for x in pending_items]

        try:
            whisper_texts = whisper_batch_transcribe(audios)
        except Exception as exc:
            report(f"  [warn] Whisper batch @idx={idx}: {exc}")
            whisper_texts = ["[ERROR]"] * len(pending_items)

        try:
            mms_texts = mms_batch_transcribe(audios)
        except Exception as exc:
            report(f"  [warn] MMS batch @idx={idx}: {exc}")
            mms_texts = ["[ERROR]"] * len(pending_items)

        for sid, gt, wtxt, mtxt in zip(ids, gt_texts, whisper_texts, mms_texts):
            new_rows.append({
                "split": split_name,
                "id": sid,
                "ground_truth": gt,
                "whisper": wtxt,
                "mms": mtxt
            })
            processed_ids.add(sid)
            processed_in_split += 1

        pending_items = []

        # ---- progress every DISPLAY_EVERY samples ----
        if processed_in_split % DISPLAY_EVERY == 0:
            report(
                f"[{split_name}] processed {processed_in_split}/{total} "
                f"(rows saved so far: {len(df) + len(new_rows)})"
            )

        # ---- periodic checkpoint ----
        if len(new_rows) >= CHECKPOINT_INTERVAL:
            df = pd.concat([df, pd.DataFrame(new_rows)], ignore_index=True)
            save_checkpoint_df(df)
            new_rows = []

    # ---- flush any leftover partial batch ----
    if pending_items:
        audios   = [x["audio"]["array"] for x in pending_items]
        gt_texts = [x["sentence"] for x in pending_items]
        ids      = [str(x["path"]) for x in pending_items]

        try:
            whisper_texts = whisper_batch_transcribe(audios)
        except Exception as exc:
            report(f"  [warn] Whisper final @idx={idx}: {exc}")
            whisper_texts = ["[ERROR]"] * len(pending_items)

        try:
            mms_texts = mms_batch_transcribe(audios)
        except Exception as exc:
            report(f"  [warn] MMS final @idx={idx}: {exc}")
            mms_texts = ["[ERROR]"] * len(pending_items)

        for sid, gt, wtxt, mtxt in zip(ids, gt_texts, whisper_texts, mms_texts):
            new_rows.append({
                "split": split_name,
                "id": sid,
                "ground_truth": gt,
                "whisper": wtxt,
                "mms": mtxt
            })
            processed_ids.add(sid)
            processed_in_split += 1

        report(
            f"[{split_name}] processed {processed_in_split}/{total} "
            f"(final flush)"
        )

    if new_rows:
        df = pd.concat([df, pd.DataFrame(new_rows)], ignore_index=True)
        save_checkpoint_df(df)

    report(f"[{split_name}] DONE — {processed_in_split} new samples in this split")
    return df, processed_ids


# ============================================================
# Run
# ============================================================
report("Initializing checkpoint...")
df, processed_ids = load_checkpoint_df()
report(f"Already processed: {len(processed_ids)} rows")

report("Testing inference on one sample...")
test_item = dataset["validation"][0]
_ = whisper_batch_transcribe([test_item["audio"]["array"]])[0]
_ = mms_batch_transcribe([test_item["audio"]["array"]])[0]
report("Test successful.")

save_state()

for split_name in ["validation", "train", "test"]:
    report(f"=== Processing split: {split_name} ===")
    df, processed_ids = process_split(split_name, df, processed_ids)
    save_state({"rows": int(len(df))})

# final export
df.to_csv(LOCAL_OUTPUT_CSV, index=False)
try:
    df.to_excel(LOCAL_OUTPUT_XLSX, index=False)
except Exception as exc:
    report(f"  [warn] final XLSX save failed: {exc}")

save_state({"rows": int(len(df))})
report("=" * 50)
report(f"ALL DONE. Rows: {len(df)}")
report(f"CSV : {LOCAL_OUTPUT_CSV}")
report(f"XLSX: {LOCAL_OUTPUT_XLSX}")


In [ ]:
# ============================================================
# Data Cleaning and Normalization
# Using Shekar Library for Persian Text Processing
# ============================================================

!pip install -q shekar openpyxl

import pandas as pd
import re
import unicodedata
from shekar import Normalizer
from shekar.preprocessing import (
    AlphabetNormalizer,
    PunctuationRemover,
    DigitNormalizer,
    SpacingNormalizer,
    RemoveDiacritics,
    RepeatedLetterNormalizer,
    ArabicUnicodeNormalizer,
    YaNormalizer,
)
import warnings
warnings.filterwarnings("ignore")

# ============================================================
# 1. Load the raw data
# ============================================================
print("Loading dataset...")
df = pd.read_csv("./raw data/asr_train_outputs_raw.csv")
print(f"Initial samples: {len(df)}")
print(f"Columns: {df.columns.tolist()}")
print(df.head())

# ============================================================
# 2. Initialize Shekar components (using Pipeline for full control)
# ============================================================
# from shekar.pipeline import Pipeline
# Custom pipeline with Shekar components
normalizer = (
            AlphabetNormalizer()
            | ArabicUnicodeNormalizer()
            | DigitNormalizer()
            | PunctuationRemover()
            | RemoveDiacritics()
            | RepeatedLetterNormalizer()
            | SpacingNormalizer()
            | YaNormalizer()
        )

# ============================================================
# 3. Define cleaning function (using Shekar for punctuation)
# ============================================================
def clean_text(text):
    """
    Complete text cleaning pipeline for ASR outputs and ground truth.
    Using Shekar for all normalization steps.
    """
    if not isinstance(text, str) or pd.isna(text):
        return ""

    # Apply Shekar normalization pipeline
    text = normalizer(text)
    return text

# ============================================================
# 4. Apply cleaning to all columns
# ============================================================
print("\nCleaning text data...")
df['ground_truth_clean'] = df['ground_truth'].apply(clean_text)
df['whisper_clean'] = df['whisper'].apply(clean_text)
df['mms_clean'] = df['mms'].apply(clean_text)

# ============================================================
# 5. Quality Control - Filter invalid samples
# ============================================================
def calculate_cer(reference, hypothesis):
    """Calculate Character Error Rate (CER)"""
    if len(reference) == 0:
        return 1.0 if len(hypothesis) > 0 else 0.0
    
    # Simple Levenshtein distance for characters
    from Levenshtein import distance as levenshtein_distance
    return levenshtein_distance(reference, hypothesis) / len(reference)

def calculate_wer(reference, hypothesis):
    """Calculate Word Error Rate (WER)"""
    ref_words = reference.split()
    hyp_words = hypothesis.split()
    
    if len(ref_words) == 0:
        return 1.0 if len(hyp_words) > 0 else 0.0
    
    from Levenshtein import distance as levenshtein_distance
    return levenshtein_distance(' '.join(ref_words), ' '.join(hyp_words)) / len(ref_words)

def is_valid_sample(row):
    """
    Comprehensive quality control for each sample.
    Returns: (is_valid, reason)
    """
    gt = row['ground_truth_clean']
    whisper = row['whisper_clean']
    mms = row['mms_clean']
    split_name = row.get('split', 'unknown')
    
    # ====== CHECK 1: Empty texts ======
    if len(gt) == 0:
        return False, "Empty ground truth", split_name
    if len(whisper) == 0:
        return False, "Empty Whisper output", split_name
    if len(mms) == 0:
        return False, "Empty MMS output", split_name
    
    # ====== CHECK 2: Length ratio (prevents extreme cases) ======
    gt_len = len(gt)
    wh_ratio = len(whisper) / gt_len
    mms_ratio = len(mms) / gt_len
    
    if wh_ratio < 0.2:
        return False, f"Whisper too short (ratio: {wh_ratio:.2f})", split_name
    if wh_ratio > 3.5:
        return False, f"Whisper too long (ratio: {wh_ratio:.2f})", split_name
    if mms_ratio < 0.2:
        return False, f"MMS too short (ratio: {mms_ratio:.2f})", split_name
    if mms_ratio > 3.5:
        return False, f"MMS too long (ratio: {mms_ratio:.2f})", split_name
    
    # ====== CHECK 3: High CER (character error rate) ======
    cer_whisper = calculate_cer(gt, whisper)
    cer_mms = calculate_cer(gt, mms)

    if cer_whisper > 0.8 and cer_mms > 0.8:
        return False, f"Both models CER too high ({cer_whisper:.2f}, {cer_mms:.2f})", split_name
    
    # ====== CHECK 4: High WER (word error rate) ======
    wer_whisper = calculate_wer(gt, whisper)
    wer_mms = calculate_wer(gt, mms)

    if wer_whisper > 1 and wer_mms > 1:
        return False, f"Both models WER too high ({wer_whisper:.2f}, {wer_mms:.2f})", split_name
    
    # ====== CHECK 5: Persian character ratio ======
    persian_chars = re.compile(r'[ابپتثجچحخدذرزژسشصضطظعغفقکگلمنوهیآئؤ]')
    gt_persian = len(persian_chars.findall(gt)) / gt_len if gt_len > 0 else 0
    
    if gt_persian < 0.15:
        return False, f"Low Persian ratio in GT ({gt_persian:.2f})", split_name
    
    return True, "Valid", split_name

# ============================================================
# 6. Apply filters
# ============================================================
print("\nFiltering invalid samples...")
valid_flags = []
reasons = []
splits = []

# Install python-Levenshtein if not available
try:
    from Levenshtein import distance as levenshtein_distance
except ImportError:
    print("Installing python-Levenshtein for CER/WER calculation...")
    !pip install -q python-Levenshtein
    from Levenshtein import distance as levenshtein_distance

for idx, row in df.iterrows():
    is_valid, reason, split_name = is_valid_sample(row)
    valid_flags.append(is_valid)
    reasons.append(reason)
    splits.append(split_name)

df['is_valid'] = valid_flags
df['filter_reason'] = reasons
df['filter_split'] = splits

# Apply filter
df_filtered = df[df['is_valid']].copy()
df_invalid = df[~df['is_valid']].copy()

print(f"Valid samples: {len(df_filtered)}")
print(f"Invalid samples removed: {len(df_invalid)}")

# ============================================================
# 7. Show statistics of removed samples
# ============================================================
print("\n" + "="*50)
print("FILTERING STATISTICS")
print("="*50)
reason_counts = df_invalid['filter_reason'].value_counts()
for reason, count in reason_counts.items():
    print(f"  {reason}: {count} samples")

print("\nBy split:")
split_counts = df_invalid['filter_split'].value_counts()
for split, count in split_counts.items():
    print(f"  {split}: {count} samples")

# ============================================================
# 8. Save invalid samples for review (with full details)
# ============================================================
print("\nSaving invalid samples for review...")
invalid_columns = ['id', 'split', 'ground_truth', 'whisper', 'mms', 
                   'ground_truth_clean', 'whisper_clean', 'mms_clean', 
                   'filter_reason', 'filter_split']

# Add CER/WER columns for invalid samples
def add_error_metrics(df_subset):
    df_subset = df_subset.copy()
    df_subset['cer_whisper'] = df_subset.apply(
        lambda row: calculate_cer(row['ground_truth_clean'], row['whisper_clean']), axis=1
    )
    df_subset['cer_mms'] = df_subset.apply(
        lambda row: calculate_cer(row['ground_truth_clean'], row['mms_clean']), axis=1
    )
    df_subset['wer_whisper'] = df_subset.apply(
        lambda row: calculate_wer(row['ground_truth_clean'], row['whisper_clean']), axis=1
    )
    df_subset['wer_mms'] = df_subset.apply(
        lambda row: calculate_wer(row['ground_truth_clean'], row['mms_clean']), axis=1
    )
    return df_subset

df_invalid_with_metrics = add_error_metrics(df_invalid)
df_invalid_with_metrics.to_csv("asr_invalid_samples_detailed.csv", index=False)
df_invalid_with_metrics.to_excel("asr_invalid_samples_detailed.xlsx", index=False)

print(f"Invalid samples saved to: asr_invalid_samples_detailed.csv")

# ============================================================
# 9. Finalize cleaned dataset
# ============================================================
final_columns = ['split', 'id', 'ground_truth_clean', 'whisper_clean', 'mms_clean']
final_df = df_filtered[final_columns].copy()
final_df.columns = ['split', 'id', 'ground_truth', 'whisper', 'mms']

# ============================================================
# 10. Save outputs
# ============================================================
print("\nSaving cleaned datasets...")
final_df.to_csv("asr_cleaned_train2.csv", index=False)
final_df.to_excel("asr_cleaned_train2.xlsx", index=False)

# ============================================================
# 11. Show sample of cleaned data
# ============================================================
print("\n" + "="*50)
print("SAMPLE CLEANED DATA")
print("="*50)
for i in range(min(5, len(final_df))):
    row = final_df.iloc[i]
    print(f"\nSample {i+1}:")
    print(f"  ID: {row['id']}")
    print(f"  Split: {row['split']}")
    print(f"  Ground Truth: {row['ground_truth']}")
    print(f"  Whisper:      {row['whisper']}")
    print(f"  MMS:          {row['mms']}")

# ============================================================
# 12. Summary report
# ============================================================
print("\n" + "="*50)
print("SUMMARY REPORT")
print("="*50)
print(f"Total samples in original dataset: {len(df)}")
print(f"Valid samples for training:         {len(final_df)} ({len(final_df)/len(df)*100:.1f}%)")
print(f"Invalid samples removed:            {len(df_invalid)} ({len(df_invalid)/len(df)*100:.1f}%)")

print("\n✅ Data cleaning and normalization complete!")

In [ ]:
import re
import pandas as pd
from pathlib import Path

# =========================
# 1) Basic Unicode cleanup
# =========================
def basic_unicode_normalize(text: str) -> str:
    if not isinstance(text, str):
        return text
    text = (
        text.replace("ي", "ی")
            .replace("ك", "ک")
            .replace("ة", "ه")
            .replace("ۀ", "هٔ")
            .replace("ـ", "")
    )
    return text


# =========================
# 2) Exceptions
# =========================
EXCEPTIONS = {
    "ماهی", "شاهی", "چاهی", "راهی", "لاهی", "سیاهی", "کوتاهی", "پناهی", "بنگاهی","گروهی","روباهی","تنبیهی","سپاهی",
    "گاهی", "نگاهی", "آهی", "پادشاهی", "واهی", "کوهی", "دهی", "آگاهی", "تباهی","گرهی","انبوهی","ساماندهی","بیگاهی","گمراهی",
    "گیاهی", "فکاهی", "الهی", "بدیهی", "نواحی", "گواهی", "خواهی", "تنهایی", "گناهی","دستگاهی","شفاهی","جاهی","شکوهی","زرهی","گاهوار","شاهوار",
    "منتهی", "وجهی", "شبهی", "جیهی", "کاهی", "تهی", "سفیهی", "تهیدست", "هیاهو", "اکراهی","ایستگاهی","خیرخواهی","بدخواهی",
    "شاهکار","شاهکاری","بدهکار","بدهکاری","کوهسار","بزهکار","بزهکاری","آهنگر","اهنگر","آهنگری","اهنگری","صبحگاهی","شامگاهی","شبانگاهی","شبگاهی",
    "هیجان", "هیولا", "هیزم", "هیبت", "هیئت", "هیس", "هیتلر" ,"هیچ", "هیچکس", "هیچ‌کس",
    "هیچ‌چیز", "هیچ‌گاه", "هیچوقت", "هیچ‌وقت", "هیچکدام", "هیچ‌کدام", "هیچیک", "سلیمانی",
    "هیچ‌یک", "هیچجا", "هیچ‌جا", "هیچگونه", "هیچ‌گونه", "هیچی", "کماکان", "بازرسی",
    "سازماندهی", "فرماندهی", "کلاهی", "اشتباهی", "اشتباهات", "آگهی", "بدهی", "عذرخواهی","تشبیهی",
    "دانشگاهی","دادگاهی","آموزشگاهی","اموزشگاهی","ورزشگاهی","دهکده","دهکدهای","کوتهی","بندرگاهی",
    "لنگرگاهی","سحرگاهی","جایگاهی","درگاهی","پرورشگاهی","آرامگاهی","آسایشگاهی","آزمایشگاهی","چراگاهی",
    "کارگاهی","پژوهشگاهی","درمانگاهی","فروشگاهی","بازداشتگاهی","نیروگاهی","پالایشگاهی","خوابگاهی","استراحتگاهی",
    "تفرجگاهی","زادگاهی","زایشگاهی","آرایشگاهی","پیرایشگاهی","اردوگاهی","اقامتگاهی","انباشتگاهی","پرتگاهی","نمایشگاهی",
    "اندوهی","گرمابهی","توجیهی",
    "ایمان", "ایشان", "ایام","سهام","بها","سها","شها","مهان","مهمان","متهمان","پیام",
    "بیام","نیام","سیاست","سیامک","قیام","بیات","حیات","آیات","ایات","روایات",
    "نشریات","تجلیات","تجربیات","عملیات","کلیات","جزییات","جزئیات","پیمان","مالیات","اتهام",
    "واقعیات","سریمان","سلیمان","باقیمانده","ادبیات","اخلاقیات","ابهام","بریتانیا","برایمان",
    "پیامد","دهات","خصوصیات","هاست","لبنیات","نیات","سیمان","پیشانی","هاتف","ناپویاست","خیام",
    "همراهی", "الهام", "پرهام","توجهی","توجهات","جهات","هاشم","هاشور","قرنهاست","کیهان","پشیمان","پشیمانی",
    "تنهاست","بنیامین","نظریات","فرضیات","استفهام","ریاست","پایمان","بازدهی","سالهاست","ریاضیات",
    "زایمان","برایتان","برایشان","ویلیام","خویشان","محتویات","بدلیات","غزلیات","میامی", "آیتان","تایتانیک","محیاشدن",
    "جنایات","صندلیات","هیات","مویشان","مویمان","مویتان","متوفیات","پیامبر","روحیات","آریامن","آریامهر",
    "باهات","باهاش","باهام","بهاش","بهام","بهات","چرندیات","کاپیتان","پیشان","تشابهات","حکایات",
    "شهامت","بیامرزد","پریشان","ضروریات","صمیمانه","اوهام","بیاموز","اینهاست","پیمانه","رادیاتور",
    "ابیات","رباعیات","فهمانده","شکایات","کشفیات","یتیمان","بیامرز","دخانیات","حدسیات","التیام","اندیشان",
    "بویتان","بویمان","بویشان","دارویمان","دارویشان","دارویتان","شبهات","تنبیهات","پیامک","تاتنهام","حذفیات","تیاتر","باهاما",
    "بیایمانی","علیمان","حکیمان","حکیمانه","مباهات","اصلیات","فرعیات","چشمهات","بیامیز","قیامت","صدایشان",
    "گلویتان","تیتان","طلاهام", "روزهای", "موهای","تنها", "اشتها","اشتهای","زهی","بها","بهای","رهای","رهایی","بخواهی", "نخواهی",

    "ماهی‌ها", "شاهی‌ها", "راهی‌ها", "سیاهی‌ها", "‌کوتاهی‌ها","انبوهی‌ها",
    "پادشاهی‌ها","آگاهی‌ها", "تباهی‌ها","فکاهی‌ها", "بدیهی‌ها", "گواهی‌ها", "تنهایی‌ها",
    "منتهی‌ها", "کاهی‌ها", "تهی‌ها", "تهیدست‌ها", "هیاهو‌ها","هیجان‌ها", "هیولا‌ها", "هیزم‌ها", "هیئت‌ها",
    "سازماندهی‌ها", "فرماندهی‌ها", "آگهی‌ها", "بدهی‌ها", "همراهی‌ها","بازدهی‌ها","هیات‌ها",

    "ایمان‌ها", "ایام‌ها","سهام‌ها","بها‌ها","مهمان‌ها","پیام‌ها",
    "سیاست‌ها","قیام‌ها","بیات‌ها","حیات‌ها","عملیات‌ها","پیمان‌ها","مالیات‌ها","اتهام‌ها",
    "سلیمانی‌ها","باقیمانده‌ها","ادبیات‌ها","ابهام‌ها","پیامد‌ها","دهات‌ها","هاست‌ها",
    "سیمان‌ها","پیشانی‌ها","خیام‌ها","الهام‌ها", "هاشم‌ها","هاشور‌ها","کیهان‌ها","پشیمان‌ها",
    "پشیمانی‌ها","بنیامین‌ها","استفهام‌ها","ریاست‌ها", "زایمان‌ها","پیامبر‌ها","کاپیتان‌ها",
    "شهامت‌ها","پیمانه‌ها","رادیاتور‌ها","التیام‌ها","پیامک‌ها","قیامت‌ها",


    "ماهیها", "شاهیها", "راهیها", "سیاهیها", "‌کوتاهیها","انبوهیها",
    "پادشاهیها","آگاهیها", "تباهیها","فکاهیها", "بدیهیها", "گواهیها", "تنهاییها",
    "منتهیها", "کاهیها", "تهیها", "تهیدستها", "هیاهوها","هیجانها", "هیولاها", "هیزمها", "هیئتها",
    "سازماندهیها", "فرماندهیها", "آگهیها", "بدهیها", "همراهیها","بازدهیها","هیاتها",

    "ایمانها", "ایامها","سهامها","بهاها","مهمانها","پیامها",
    "سیاستها","قیامها","بیاتها","حیاتها","عملیاتها","پیمانها","مالیاتها","اتهامها",
    "سلیمانیها","باقیماندهها","ادبیاتها","ابهامها","پیامدها","دهاتها","هاستها",
    "سیمانها","پیشانیها","خیامها","الهامها", "هاشمها","هاشورها","کیهانها","پشیمانها",
    "پشیمانیها","بنیامینها","استفهامها","ریاستها", "زایمانها","پیامبرها","کاپیتانها",
    "شهامتها","پیمانهها","رادیاتورها","التیامها","پیامکها","قیامتها",

    "ماهی‌های", "شاهی‌های", "راهی‌های", "سیاهی‌های", "‌کوتاهی‌های","انبوهی‌های",
    "پادشاهی‌های","آگاهی‌های", "تباهی‌های","فکاهی‌های", "بدیهی‌های", "گواهی‌های", "تنهایی‌های",
    "منتهی‌های", "کاهی‌های", "تهی‌های", "تهیدست‌های", "هیاهو‌های","هیجان‌های", "هیولا‌های", "هیزم‌های", "هیئت‌های",
    "سازماندهی‌های", "فرماندهی‌های", "آگهی‌های", "بدهی‌های", "همراهی‌های","بازدهی‌های","هیات‌های",

    "ایمان‌های", "ایام‌های","سهام‌های","بها‌های","مهمان‌های","پیام‌های",
    "سیاست‌های","قیام‌های","بیات‌های","حیات‌های","عملیات‌های","پیمان‌های","مالیات‌های","اتهام‌های",
    "سلیمانی‌های","باقیمانده‌های","ادبیات‌های","ابهام‌های","پیامد‌های","دهات‌های","هاست‌های",
    "سیمان‌های","پیشانی‌های","خیام‌های","الهام‌های", "هاشم‌های","هاشور‌های","کیهان‌های","پشیمان‌های",
    "پشیمانی‌های","بنیامین‌های","استفهام‌های","ریاست‌های", "زایمان‌های","پیامبر‌های","کاپیتان‌های",
    "شهامت‌های","پیمانه‌های","رادیاتور‌های","التیام‌های","پیامک‌های","قیامت‌های",

    "ماهیهای", "شاهیهای", "راهیهای", "سیاهیهای", "‌کوتاهیهای","انبوهیهای",
    "پادشاهیهای","آگاهیهای", "تباهیهای","فکاهیهای", "بدیهیهای", "گواهیهای", "تنهاییهای",
    "منتهیهای", "کاهیهای", "تهیهای", "تهیدستهای", "هیاهوهای","هیجانهای", "هیولاهای", "هیزمهای", "هیئتهای",
    "سازماندهیهای", "فرماندهیهای", "آگهیهای", "بدهیهای", "همراهیهای","بازدهیهای","هیاتهای",

    "ایمانهای", "ایامهای","سهامهای","بهاهای","مهمانهای","پیامهای",
    "سیاستهای","قیامهای","بیاتهای","حیاتهای","عملیاتهای","پیمانهای","مالیاتهای","اتهامهای",
    "سلیمانیهای","باقیماندههای","ادبیاتهای","ابهامهای","پیامدهای","دهاتهای","هاستهای",
    "سیمانهای","پیشانیهای","خیامهای","الهامهای", "هاشمهای","هاشورهای","کیهانهای","پشیمانهای",
    "پشیمانیهای","بنیامینهای","استفهامهای","ریاستهای", "زایمانهای","پیامبرهای","کاپیتانهای",
    "شهامتهای","پیمانههای","رادیاتورهای","التیامهای","پیامکهای","قیامتهای",

    "آهنگرها","اهنگرها""آهنگر‌ها","اهنگر‌ها","آهنگرهای","اهنگرهای""آهنگر‌های","اهنگر‌های","آهنگرهایی",
    "اهنگرهایی""آهنگر‌هایی","اهنگر‌هایی","آهنگران","اهنگران","انتها","انتهای","انتهایی",
    

    "بهتر","بهتری","بهترین","بهترینی","بهسازی","بهداری","بهدار","بهزیستی",
    "کلاهبردار","کلاهبرداری","کلاهبردارها","کلاهبردار‌ها","کلاهبردارهای","کلاهبردار‌های",
    "گناهکار","گناهکاری","گناهکاران","گناهکارها","گناهکار‌ها","گناهکاریها","گناهکاری‌ها","گناهکاریهای","گناهکاری‌های",
    "نگهداری","نگهدار","خدانگهدار","خدا‌نگهدار","خدانگهداری",

    "پیشای","رهبر","رهبری","شاهنامه","شاهنامه‌ی","تشابهی","کوهستان","کوهستانی","سماور","باور","لهستان","لهستانی","مشهور","جمهور","جمهوری",
}


# =========================
# 3) Regex patterns
# =========================

# 3.0) الگوی عمومی برای «های / ها» و پسوندهای متصل به آن (فعالیتهای -> فعالیت‌های)
PATTERN_GENERAL_HAY = re.compile(
    r"\b([آ-یءٔ]{2,})(هایمان|هایتان|هایشان|هایم|هایت|هایش|هایی|های|ها)\b"
)

# 3.1) ه + other suffixes
PATTERN_HE_OTHER = re.compile(
    r"\b([آ-یءٔ]+ه)(ام|ات|اش|ایم|اید|اند|است|مان|تان|شان|ای|بود|شد|باش|شو)\b"
)

# 3.2) ه + ی (نکتهی -> نکته‌ی)
PATTERN_HE_EZAFE = re.compile(r"\b([آ-یءٔ]{2,}ه)ی\b")

# 3.3) Compound suffixes
COMPOUND_SUFFIXES = [
    "گیری","گیر","گیران","گر","گری", "گران", "گرانه","گرای","گرایی",
    "گزار", "گزاری", "گزاران",
    "گذار","گذاری","گذاران",
    "سازی", "ساز","سازان","سازندگان",
    "داری","دار","داران",
    "بندی", "بند",
    "کشی", "کش","کشان",
    "آمیز", "آمیزی","آمیزان",
    "شناسی", "شناس","شناسان",
    "پذیری","پذیر","پذیران","پذیرندگان",
    "ستانی","ستان","ستانان",
    "رانی","رانان",
    "پروری","پرور","پروران","پرورندگان",
    "بازی","باز",
    "بازکن","بازکنی","بازکنان",
    "یاب","یابی","یابندگان",
    "پیما","پیمایی",
    "آور","آوری","آورندگان",
    "برداری","بردار", 
    "بر","بری", "بران",
    "دست","دستی","دستان",
    "سار", "شهر","آباد" ,"خانه","سرا","کده",
    "انداز","اندازی","اندازان",
    "افزار", "افزاری", "ابزار","ابزاری",
    "دار","داری","در","دری",
    "طلب", "طلبی","طلبان","طلبها","طلب‌ها",
    "پزی", "پز","پزان","پزها","پز‌ها",
    "انگیز","انگیزی","انگیزان",
    "افزا","افزایی","افزایان",
    "افروز","افروزی","افروزان",
    "افکن","افکنی","افکنان",
    "پراکن","پراکنی","پراکنان",
    "فرسا","فرسایی","فرسایان",
    "فشان","فشانی","فشانان","فشانها","فشان‌ها",
    "ریز","ریزی","ریزان","ریزها","ریز‌ها",
    "خیز","خیزی","خیزان","خیزها","خیز‌ها",
    "خواه", "خواهی","خواهان","خواهندگان",
    "کن","کنی","کننده", "کنندگان","کننده‌ها",
    "دهی","دهنده","دهان","دهندگان",
    "روی","رو","رونده","روندگان",
    "آلود","آلودی","آلودگان",
    "آسا","آسای","آسایی","آسایان",
    "نگر", "نگری","نگران","نگرها","نگر‌ها",
    "بین","بینی","بینان","بینها","بین‌ها",
    "اندیش","اندیشی","اندیشان","اندیشها","اندیش‌ها",
    "فکر","فکری","فکرانه", "فکران","فکرها","فکر‌ها",
    "نویس","نویسی","نویسان", "نویسها","نویس‌ها",
    "نگار","نگاری","نگاران","نگارها","نگار‌ها",
    "پرداز","پردازی","پردازان","پردازها","پرداز‌ها",
    "دان","دانی","دانان","دانها","دان‌ها",
    "آزما","آزمایی","آزمایان",
    "گو","گوی","گویی","گویان",
    "کاری","کار", "کاران","کارها","کار‌ها",
    "پیشه","پیشگی", "پیشگان","پیشه‌ها",
    "مند","مندی","مندان","مندها","مند‌ها",
    "خوار","خواری","خواران", "خور","خوری", "خورده","خوردگی","خوردگان",
    "نوش","نوشی","نوشان","نوشها","نوش‌ها",
    "نشین","نشینی","نشینان","نشینها","نشین‌ها",
    "پوش","پوشی","پوشان","پوشها","پوش‌ها",
    "فروش","فروشی","فروشان","فروشها","فروش‌ها",
    "ربا","ربایی","ربایان","رباها","ربا‌ها",
    "کوب","کوبی","کوبان","کوبها","کوب‌ها",
    "بخش","بخشی","بخشان",
    "لوح","لوحی","لوحها","لوح‌ها","لوحان","لوحانه","لوحانه‌ای",
    "دوست","دوستی","دوستان","دوستها","دوست‌ها",
    "وار","واری","واران",
    "ور","وری","وران",
    "زن","زنان","زنی","زده",
    "جو","جویی","جویان",
    "زدا","زدایی","زدایان",
    "دم","دمی", "دمان",
    "ناک", "ناکی","ناکان",
    "گین","گینی","گینان",
    "توز","توزی","توزانه",
    "بند", "بندی",
    "تر","ترین",
    "مال","مالی",
    "نظام", "نظامی",
    "نامه",
    "فرنگی",
]

PATTERN_COMPOUND = re.compile(
    r"\b([آ-یءٔ]+[ه])(" + "|".join(map(re.escape, COMPOUND_SUFFIXES)) + r")\b"
)

ALEF_COMPOUND_SUFFIXES = [
    "آمیز", "آمیزی", "آمیزان",
    "آور", "آوری", "آورندگان",
    "آباد", 
    "انداز", "اندازی", "اندازان",
    "افزار", "افزاری", "ابزار", "ابزاری",
    "انگیز", "انگیزی", "انگیزان",
    "افزا", "افزایی", "افزایان",
    "افروز", "افروزی", "افروزان",
    "افکن", "افکنی", "افکنان",
    "آلود", "آلودی", "آلودگان",
    "آسا", "آسای", "آسایی", "آسایان",
    "اندیش", "اندیشی", "اندیشان",
    "آزما", "آزمایی", "آزمایان",
    "آموز","آموزی","آموزان",
]

PATTERN_ALEF_COMPOUND = re.compile(
    r"\b([آ-یءٔ]+)(" + "|".join(map(re.escape, ALEF_COMPOUND_SUFFIXES)) + r")\b"
)

# 3.4) Fixed words / frequent patterns
PATTERN_HAMEH = re.compile(r"\bهمه(?=[آ-ی])")
PATTERN_FI_AL = re.compile(r"\bفیال(?=[آ-ی])")
PATTERN_BEYN_AL = re.compile(r"\bبینال(?=[آ-ی])")


# 3.5) Prepositions
BE_ZWNJ_FORMS = {
    "معنی", "ویژه", "جای", "خاطر", "نظر", "طور", "جز", "مراتب",
    "تدریج", "مثابه", "سبب", "ندرت", "ناچار", "اندازه", "جانب",
    "عنوان", "خصوص", "عمد", "سهو", "واقع", "ظاهر", "زحمت", "اجبار",
    "اختصار", "اصطلاح", "دلخواه", "دنبال","عقبه" ,"اثر", "کلی","زودی",
    "تفصیل","حالت", "هنگام","مصداق"
}

BE_SPACE_FORMS = {
    "سر", "دست", "طرف", "منزل", "خانه", "جنگ", "راه", "زور", "کار",
    "حال", "دل", "جان", "سمت", "فکر", "چاه", "کس", "چیز", "من",
    "تو", "او", "ما", "شما", "آنها", "ایشان", "نام", "دلیل", "علت","طرز"
    "محض", "عقب", "پای", "بالا", "پایین", "کنار", "داخل", "خارج","شکل"
    "رو", "زیر", "موقع" , "پیگیری", "خطر", "استطاعت","هم","عهده","سوی",
    "صورت","آسانی","سختی","منصه","منصه‌ی","آرامی","ارامی","سرعت","راستی",
    "وجود","دنیا","آهستگی","اهستگی","صرفه","وثیقه",
}

CHE_SPACE_FORMS = {
    "کس", "کسی", "چیز", "وقت", "موقع", "زمان", "زمانی", "طور", "گونه",
    "قدر", "اندازه", "تعداد", "نام", "رنگ", "کار", "شغل", "شهر",
    "کشور", "روز", "ساعت", "ساعتی", "سال", "ماه", "فکری", "حرفی",
    "خبری", "دلیلی", "منظوری", "حسی", "نسبتی", "شماره", "قیمتی",
    "مشکلی", "اتفاقی", "کلاسی", "کتابی", "درسی", "غذایی", "لباسی",
    "مدلی", "برنامه‌ای", "راهی", "نوع"
}

# 3.6) Safe whitelist
WHITELIST = {
    "بهینهای":"بهینه‌ای",
    "فایدهای":"فایده‌ای",
    "ذرهای":"ذره‌ای",
    "جامعهای":"جامعه‌ای",
    "عدهای":"عده‌ای",
    "اندازهای":"اندازه‌ای",
    "بچهای":"بچه‌ای",
    "غریبهای":"غریبه‌ای",
    "قهوهای":"قهوه‌ای",
    "ویژهای":"ویژه‌ای",
    "اجارهای":"اجاره‌ای",
    "اجازهای":"اجازه‌ای",
    "چهرهای":"چهره‌ای",
    "خانهای":"خانه‌ای",
    "کلبهای":"کلبه‌ای",
    "برکهای":"برکه‌ای",
    "دریاچهای":"دریاچه‌ای",
    "رودخانهای":"رودخانه‌ای",
    "حصهای":"حصه‌ای",
    "ورطهای":"ورطه‌ای",
    "خوشمزهای":"خوشمزه‌ای",
    "بدمزهای":"بدمزه‌ای",
    "مزهای":"مزه‌ای",
    "افشانهای":"افشانه‌ای",
    "آمیزهای":"آمیزه‌ای",
    "نهفتهای":"نهفته‌ای",
    "عقیدهای":"عقیده‌ای",
    "تپهای":"تپه‌ای",
    "خاطرهای":"خاطره‌ای",
    "چارهای":"چاره‌ای",
    "شایعهای":"شایعه‌ای",
    "مبارزهای":"مبارزه‌ای",
    "کرایهای":"کرایه‌ای",
    "گرسنهای":"گرسنه‌ای",
    "گشنهای":"گشنه‌ای",
    "تشنهای":"تشنه‌ای",
    "حافظهای":"حافظه‌ای",
    "نقشهای":"نقشه‌ای",
    "توطئهای":"توطئه‌ای",
    "حاشیهای":"حاشیه‌ای",
    "نکتهای":"نکته‌ای",
    "مسئلهای":"مسئله‌ای",
    "مسیلهای":"مسیله‌ای",
    "نقیضهای":"نقیضه‌ای", #
    "اندیشهای":"اندیشه‌ای",
    "نخبهای":"نخبه‌ای",
    "حماسهای":"حماسه‌ای",
    "دادهای":"داده‌ای",
    "مجموعهای":"مجموعه‌ای",
    "جادهای":"جاده‌ای",
    "صخرهای":"صخره‌ای",
    "میلهای":"میله‌ای",
    "اشارهای":"اشاره‌‌ای",
    "دهکدهای":"دهکده‌ای",
    "دریچهای":"دریچه‌ای",
    "فرخندهای":"فرخنده‌ای",
    "خندهای":"خنده‌ای",
    "گریهای":"گریه‌ای",
    "تودهای":"توده‌ای",
    "شاهزادهای":"شاهزاده‌ای",
    "سالیانهای":"سالیانه‌ای",
    "ماهیانهای":"ماهیانه‌ای",
    # "بردهای":"برده‌ای",
    "عمدهای":"عمده‌ای",
    "زندهای":"زنده‌ای",
    "روزنامهای":"روزنامه‌ای",
    "کیسهای":"کیسه‌ای",
    "فراوردهای":"فراورده‌ای",
    "فرآوردهای":"فرآورده‌ای",
    "سوزندهای":"سوزنده‌ای",
    "پنبهای":"پنبه‌ای",
    "خیره‌کنندهای":"خیره‌کننده‌ای",
    "کنندهای":"کننده‌ای",
    "بیفایدهای":"بی‌فایده‌ای",
    "برجستهای":"برجسته‌ای",
    "بشکهای":"بشکه‌ای",
    "پشهای":"پشه‌ای",
    "خربزهای":"خربزه‌ای",
    "هندوانهای":"هندوانه‌ای",
    "بودهای":"بوده‌ای",
    "بودهام":"بوده‌ام",
    "بودهاید":"بوده‌اید",
    "بودهایم":"بوده‌ایم",
    "بودهاست":"بوده‌است",
    "بودهاند":"بوده‌اند",
    "کردهام":"کرده‌ام",
    "کردهاست":"کرده‌است",
    "کردهایم":"کرده‌ایم",
    "کردهاید":"کرده‌اید",
    "کردهاند":"کرده‌اند",
    "آمدهایم":"آمده‌ایم",
    "آمدهاید":"آمده‌اید",
    "آمدهاند":"آمده‌اند",
    "آمدهام":"آمده‌ام",
    "آمدهای":"آمده‌ای",
    "‌آمدهاست":"آمده‌است",
    "لکهای":"لکه‌ای",
    "انگیزهای":"انگیزه‌ای",
    "هفتهای":"هفته‌ای",
    "درختچهای":"درختچه‌ای",
    "بوتهای":"بوته‌ای",
    "پروردهای":"پرورده‌ای",
    "فیتیلهای":"فیتیله‌ای",
    "تیلهای":"تیله‌ای",
    "تفرقهای":"تفرقه‌ای",
    "دانهای":"دانه‌ای",
    "بهانهای":"بهانه‌ای",
    "ترانهای":"ترانه‌ای",
    "مردانهای":"مردانه‌ای",
    "زنانهای":"زنانه‌ای",
    "بچگانهای":"بچگانه‌ای",
    "کودکانهای":"کودکانه‌ای",
    "میوهای":"میوه‌ای",
    "درهای":"دره‌ای",
    "قلهای":"قله‌ای",
    "سکهای":"سکه‌ای",
    "خستهای":"خسته‌ای",
    "یافتهای":"یافته‌ای",
    "مقالهای":"مقاله‌ای",
    "مقدمهای":"مقدمه‌ای",
    "ناحیهای":"ناحیه‌ای",
    "مطالعهای":"مطالعه‌ای",
    "مصاحبهای":"مصاحبه‌ای",
    "محاسبهای":"محاسبه‌ای",
    "معاینهای":"معاینه‌ای",
    "مشاهدهای":"مشاهده‌ای",
    "نظریهای":"نظریه‌ای",
    "فرضیهای":"فرضیه‌ای",
    "مداخلهای":"مداخله‌ای",
    "مذاکرهای":"مذاکره‌ای",
    "مکاتبهای":"مکاتبه‌ای",
    "معاملهای":"معامله‌ای",
    "مقابلهای":"مقابله‌ای",
    "مسابقهای":"مسابقه‌ای",
    "افسانهای":"افسانه‌ای",
    "افسردهای":"افسرده‌ای",
    "مژدهای":"مژده‌ای",
    "ژولیدهای":"ژولیده‌ای",
    "آلودهای":"آلوده‌ای",
    "آزمودهای":"آزموده‌ای",
    "آزردهای":"آزرده‌ای",
    "آفریدهای":"آفریده‌ای",
    "آمیختهای":"آمیخته‌ای",
    "سادهای":"ساده‌ای",
    "آراستهای":"آراسته‌ای",
    "پیراستهای":"پیراسته‌ای",
    "خواستهای":"خواسته‌ای",
    "تنیدهای":"تنیده‌ای",
    "چکیدهای":"چکیده‌ای",
    "خلاصهای":"خلاصه‌ای",
    "تازهای":"تازه‌ای",
    "کاشانهای":"کاشانه‌ای",
    "لانهای":"لانه‌ای",
    "آشیانهای":"آشیانه‌ای",
    "کتابخانهای":"کتابخانه‌ای",
    "قهوه‌خانهای":"قهوه‌خانه‌ای",
    "میخانهای":"میخانه‌ای",
    "گلخانهای":"گلخانه‌ای",
    "آشپزخانهای":"آشپزخانه‌ای",
    "سردخانهای":"سردخانه‌ای",
    "مهمانخانهای":"مهمان‌خانه‌ای",
    "زورخانهای":"زورخانه‌ای",
    "آتشخانهای":"آتشخانه‌ای",
    "چایخانهای":"چایخانه‌ای",
    "کارخانهای":"کارخانه‌ای",
    "مکتبخانهای":"مکتبخانه‌ای",
    "داروخانهای":"داروخانه‌ای",
    "بیشهای":"بیشه‌ای",
    "شانهای":"شانه‌ای",
    "دکمهای":"دکمه‌ای",
    "حوصلهای":"حوصله‌ای",
    "آیندهای":"آینده‌ای",
    "زاویهای":"زاویه‌ای",
    "فاجعهای":"فاجعه‌ای",
    "سانحهای":"سانحه‌ای",
    "واقعهای":"واقعه‌ای",
    "مدرسهای":"مدرسه‌ای",
    "کاسهای":"کاسه‌ای",
    "جرعهای":"جرعه‌ای",
    "قابلمهای":"قابلمه‌ای",
    "شیشهای":"شیشه‌ای",
    "کوزهای":"کوزه‌ای",
    "ماهیتابهای":"ماهیتابه‌ای",
    "خامهای":"خامه‌ای",
    "پروانهای":"پروانه‌ای",
    "موریانهای":"موریانه‌ای",
    "مورچهای":"مورچه‌ای",
    "عاقلانهای":"عاقلانه‌ای",
    "حکیمانهای":"حکیمانه‌ای",
    "عاشقانهای":"عاشقانه‌ای",
    "شجاعانهای":"شجاعانه‌ای",
    "دوستانهای":"دوستانه‌ای",
    "خصمانهای":"خصمانه‌ای",
    "عارفانهای":"عارفانه‌ای",
    "شاعرانهای":"شاعرانه‌ای",
    "عالمانهای":"عالمانه‌ای",
    "جاهلانهای":"جاهلانه‌ای",
    "ظالمانهای":"ظالمانه‌ای",
    "عادلانهای":"عادلانه‌ای",
    "منصفانهای":"منصفانه‌ای",
    "مغرضانهای":"مغرضانه‌ای",
    "صادقانهای":"صادقانه‌ای",
    "مباحثهای":"مباحثه‌ای",
    "مزایدهای":"مزایده‌ای",
    "مناقشهای":"مناقشه‌ای",
    "وعدهای":"وعده‌ای",
    "رایحهای":"رایحه‌ای",
    "حادثهای":"حادثه‌ای",
    "مزرعهای":"مزرعه‌ای",
    "معاهدهای":"معاهده‌ای",
    "مرحلهای":"مرحله‌ای",
    "دههای":"دهه‌ای",
    "دورهای":"دوره‌ای",
    "لحظهای":"لحظه‌ای",
    "ثانیهای":"ثانیه‌ای",
    "دقیقهای":"دقیقه‌ای",
    "نالهای":"ناله‌ای",
    "آرمیدهای":"آرمیده‌ای",
    "افروختهای":"افروخته‌ای",
    "فروختهای":"فروخته‌ای",
    "انباشتهای":"انباشته‌ای",
    "گستردهای":"گسترده‌ای",
    "ارهای":"اره‌ای",
    "انگارهای":"انگاره‌ای",
    "باغچهای":"باغچه‌ای",
    "غنچهای":"غنچه‌ای",
    "قصهای":"قصه‌ای",
    "غصهای":"غصه‌ای",
    "بامیهای":"بامیه‌ای",
    "بادیهای":"بادیه‌ای",
    "بازیچهای":"بازیچه‌ای",
    "بازندهای":"بازنده‌ای",
    "برهای":"بره‌ای",
    "افتادهای":"افتاده‌ای",
    "بیچارهای":"بیچاره‌ای",
    "درماندهای":"درمانده‌ای",
    "بیگانهای":"بیگانه‌ای",
    "بیهودهای":"بیهوده‌ای",
    "پرندهای":"پرنده‌ای",
    "پردهای":"پرده‌ای",
    "پونهای":"پونه‌ای",
    "پیادهای":"پیاده‌ای",
    "پنجرهای":"پنجره‌ای",
    "پوشهای":"پوشه‌ای",
    "پیالهای":"پیاله‌ای",
    "جملهای":"جمله‌ای",
    "جعبهای":"جعبه‌ای",
    "جزیرهای":"جزیره‌ای",
    "جلگهای":"جلگه‌ای",
    "تکهای":"تکه‌ای",
    "جوجهای":"جوجه‌ای",
    "حوالهای":"حواله‌ای",
    "حوزهای":"حوزه‌ای",
    "حجرهای":"حجره‌ای",
    "دلمهای":"دلمه‌ای",
    "ذخیرهای":"ذخیره‌ای",
    "توشهای":"توشه‌ای",
    "رسانهای":"رسانه‌ای",
    "ژالهای":"ژاله‌ای",
    "سرچشمهای":"سرچشمه‌ای",
    "سایهای":"سایه‌ای",
    "شیوهای":"شیوه‌ای",
    "صفحهای":"صفحه‌ای",
    "عرصهای":"عرصه‌ای",
    "عصارهای":"عصاره‌ای",
    "علامهای":"علامه‌ای",
    "فرزانهای":"فرزانه‌ای",
    "فرشتهای":"فرشته‌ای",
    "مائدهای":"مائده‌ای",
    "مایدهای":"مایده‌ای",
    "فرستندهای":"فرستنده‌ای",
    "گیرندهای":"گیرنده‌ای",
    "فشردهای":"فشرده‌ای",
    "قبلهای":"قبله‌ای",
    "قافیهای":"قافیه‌ای",
    "کافهای":"کافه‌ای",
    "کینهای":"کینه‌ای",
    "گوجهای":"گوجه‌ای",
    "مژهای":"مژه‌ای",
    "وسیلهای":"وسیله‌ای",
    "یابندهای":"یابنده‌ای",
    "جویندهای":"جوینده‌ای",
    "رانندهای":"راننده‌ای",
    "دوندهای":"دونده‌ای",
    "بینندهای":"بیننده‌ای",
    "نوازندهای":"نوازنده‌ای",
    "خزندهای":"خزنده‌ای",
    "جهندهای":"جهنده‌ای",
    "پردازندهای":"پردازنده‌ای",
    "فروشندهای":"فروشنده‌ای",
    "دوزندهای":"دوزنده‌ای",
    "ویرانهای":"ویرانه‌ای",
    "کوهپایهای":"کوهپایه‌ای",
    "شبه‌جزیرهای":"شبه‌جزیره‌ای",
    "تنگهای":"تنگه‌ای",
    "کوفتهای":"کوفته‌ای",
    "گویندهای":"گوینده‌ای",
    "شنوندهای":"شنونده‌ای",
    "خوانندهای":"خواننده‌ای",
    "نویسندهای":"نویسنده‌ای",
    "سازندهای":"سازنده‌ای",
    "پرسندهای":"پرسنده‌ای",
    "آویختهای":"آویخته‌ای",
    "حسینیهای":"حسینیه‌ای",
    "فاطمیهای":"فاطمیه‌ای",
    "تکیهای":"تکیه‌ای",
    "قضیهای":"قضیه‌ای",
    "شنبهای":"شنبه‌ای",
    "یکشنبهای":"یکشنبه‌ای",
    "دوشنبهای":"دوشنبه‌ای",
    "سه‌شنبهای":"سه‌شنبه‌ای",
    "چهارشنبهای":"چهارشنبه‌ای",
    "پنج‌شنبهای":"پنج‌شنبه‌ای",
    "آدینهای":"آدینه‌ای",
    "آینهای":"آینه‌ای",
    "واژهای":"واژه‌ای",
    "دوچرخهای":"دوچرخه‌ای",
    "سه‌چرخهای":"سه‌چرخه‌ای",
    "ماسهای":"ماسه‌ای",
    "دایرهای":"دایره‌ای",
    "خمرهای":"خمره‌ای",
    "ایستادهای":"ایستاده‌ای",
    "نوشتهای":"نوشته‌ای",
    "شنیدهای":"شنیده‌ای",
    "نمرهای":"نمره‌ای",
    "قاعدهای":"قاعده‌ای",
    "تجربهای":"تجربه‌ای",
    "جاذبهای":"جاذبه‌ای",
    "دافعهای":"دافعه‌ای",
    "مادهای":"ماده‌ای",
    "لهجهای":"لهجه‌ای",
    "سهمیهای":"سهمیه‌ای",
    "هزینهای":"هزینه‌ای",
    "نتیجهای":"نتیجه‌ای",
    "رزومهای":"رزومه‌ای",
    "شجره‌نامهای":"شجره‌نامه‌ای",
    "انگیزه‌نامهای":"انگیزه‌نامه‌ای",
    "خونهای":"خونه‌ای",
    "چهارخانهای":"چهارخانه‌ای",
    "چهارخونهای":"چهارخونه‌ای",
    "شیفتهای":"شیفته‌ای",
    "فریفتهای":"فریفته‌ای",
    "مجسمهای":"مجسمه‌ای",
    "نگارهای":"نگاره‌ای",
    "سجادهای":"سجاده‌ای",
    "طرهای":"طره‌ای",
    "تیشهای":"تیشه‌ای",
    "درخشندهای":"درخشنده‌ای",
    "برنامهای":"برنامه‌ای",
    "گونهای":"گونه‌ای",
    "جهیزیهای":"جهیزیه‌ای",
    "حیلهای":"حیله‌ای",
    "برهنهای":"برهنه‌ای",
    "رشتهای":"رشته‌ای",
    "پهنهای":"پهنه‌ای",
    "اعجوبهای":"اعجوبه‌ای",
    "خانوادهای":"خانواده‌ای",
    "نوهای":"نوه‌ای",
    "خیمهای":"خیمه‌ای",
    "روزانهای":"روزانه‌ای",
    "فرهیختهای":"فرهیخته‌ای",
    "شایستهای":"شایسته‌ای",
    "ادارهای":"اداره‌ای",
    "شعبهای":"شعبه‌ای",
    "شعبدهای":"شعبده‌ای",
    "پدیدهای":"پدیده‌ای",
    "روحیهای":"روحیه‌ای",
    "خوشهای":"خوشه‌ای",
    "بیمهنامهای":"بیمه‌نامه‌ای",
    "علاقهای":"علاقه‌ای",
    "ملاقهای":"ملاقه‌ای",
    "وسوسهای":"وسوسه‌ای",
    "بدجنسانهای":"بدجنسانه‌ای",
    "ریاکارانهای":"ریاکارانه‌ای",
    "جانانهای":"جانانه‌ای",
    "دوگانهای":"دوگانه‌ای",
    "چرتکهای":"چرتکه‌ای",
    "محافظهکارانهای":"محافظه‌کارانه‌ای",
    "وهلهای":"وهله‌ای",
    "برشتهای":"برشته‌ای",
    "هستهای":"هسته‌ای",
    "گوشوارهای":"گوشواره‌ای",
    "نردهای":"نرده‌ای",
    "نسخهای":"نسخه‌ای",
    "دروازهای":"دروازه‌ای",
    "ستارهای":"ستاره‌ای",
    "سیارهای":"سیاره‌ای",
    "طاقچهای":"طاقچه‌ای",
    "بیلچهای":"بیلچه‌ای",
    "گشودهای":"گشوده‌ای",
    "شعلهای":"شعله‌ای",
    "حوضچهای":"حوضچه‌ای",
    "پاشنهای":"پاشنه‌ای",
    "هدیهای":"هدیه‌ای",
    "جایزهای":"جایزه‌ای",
    "جنگندهای":"جنگنده‌ای",
    "کلوچهای":"کلوچه‌ای",
    "نوشابهای":"نوشابه‌ای",
    "قطرهای":"قطره‌ای",
    "خرقهای":"خرقه‌ای",
    "سویهای":"سویه‌ای",
    "سینهای":"سینه‌ای",
    "کلیهای":"کلیه‌ای",
    "شکوهای":"شکوه‌ای",
    "شکوفهای":"شکوفه‌ای",
    "یارانهای":"یارانه‌ای",
    "رایانهای":"رایانه‌ای",
    "قدیسهای":"قدیسه‌ای",
    "واهمهای":"واهمه‌ای",
    "غیرمنتظرهای":"غیرمنتظره‌ای",
    "پیرایهای":"پیرایه‌ای",
    "ترشمزهای":"ترشمزه‌ای",
    "مایهای":"مایه‌ای",
    "حملهای":"حمله‌ای",
    "ماهوارهای":"ماهواره‌ای",
    "سفینهای":"سفینه‌ای",
    "موزهای":"موزه‌ای",
    "نیمکرهای":"نیمکره‌ای",
    "مخفیانهای":"مخفیانه‌ای",
    "پیچیدهای":"پیچیده‌ای",
    "کهنهای":"کهنه‌ای",
    "جریمهای":"جریمه‌ای",
    "توسعهای":"توسعه‌ای",
    "تسویهای":"تسویه‌ای",
    "تصفیهای":"تصفیه‌ای",
    "شبههای":"شبهه‌ای",
    "مبهوت‌کنندهای":"مبهوت‌کننده‌ای",
    "فیلمنامهای":"فیلمنامه‌ای",
    "نمایشنامهای":"نمایشنامه‌ای",
    "محرمانهای":"محرمانه‌ای",
    "دانشکدهای":"دانشکده‌ای",
    "مغازهای":"مغازه‌ای",
    "پارچهای":"پارچه‌ای",
    "شیرجهای":"شیرجه‌ای",
    "دندهای":"دنده‌ای",
    "جلسهای":"جلسه‌ای",
    "منظرهای":"منظره‌ای",
    "سنجیدهای":"سنجیده‌ای",
    "شبانهای":"شبانه‌ای",
    "دفعهای":"دفعه‌ای",
    "رژهای":"رژه‌ای",
    "اعلامیهای":"اعلامیه‌ای",
    "برآمدهای":"برآمده‌ای",
    "مبالغهای":"مبالغه‌ای",
    "کنایهای":"کنایه‌ای",
    "جمجمهای":"جمجمه‌ای",
    "جداگانهای":"جداگانه‌ای",
    "جنبهای":"جنبه‌ای",
    "نقطهای":"نقطه‌ای",
    "محاصرهای":"محاصره‌ای",
    "رشوهای":"رشوه‌ای",
    "زبدهای":"زبده‌ای",
    "مقایسهای":"مقایسه‌ای",
    "روزمرهای":"روزمره‌ای",
    "مطالبهای":"مطالبه‌ای",
    "اندوختهای":"اندوخته‌ای",
    "راهبهای":"راهبه‌ای",
    "تسمهای":"تسمه‌ای",
    "بیمزهای":"بی‌مزه‌ای",
    "برنامهای":"برنامه‌ای",
    "لطیفهای":"لطیفه‌ای",
    "جبههای":"جبهه‌ای",
    "محکمهای":"محکمه‌ای",
    "ماهیچهای":"ماهیچه‌ای",
    "تولهای":"توله‌ای",
    "زبالهای":"زباله‌ای",
    "خرخندهای":"خرخنده‌ای",
    "اشاعهای":"اشاعه‌ای",
    "خامنهای":"خامنه‌ای",
    "کلیشهای":"کلیشه‌ای",
    "اژهای":"اژه‌ای",
    "پیلهای":"پیله‌ای",




    "شاءن" : "شأن",
    "بالاءخره" : "بالأخره",
    "فنآوری":"فناوری",
    "پایاننامه":"پایان‌نامه",
    "باریک‌بینانهای":"باریک‌بینانه‌ای",
    "بیفایده":"بی‌فایده",
    "بیایمانی":"بی‌ایمانی",
    "بیانگیزگی":"بی‌انگیزگی",
    "بیمهری":"بی‌مهری",
    "روشنبینانه":"روشن‌بینانه",
    "ککمک":"کمک",
    "بیاحتیاطی":"بی‌احتیاطی",
    "چشمچرانی":"چشم‌چرانی",
    "صرفنظر":"صرف‌نظر",
    "اسبسواری":"اسب‌سواری",
    "نخستوزیر":"نخست‌وزیر",
    "خشخش":"خش‌خش",
    "بنبست":"بن‌بست",
    "حسننیت":"حسن‌نیت",
    "پیشپرتابی":"پیش‌پرتابی",
    "آذینبندی":"آذین‌بندی",
    "گزارشگیری":"گزارش‌گیری",
    "پاداشده‌ی":"پاداش‌دهی",
    "پاداشدهی":"پاداش‌دهی",
    "تخممرغ":"تخم‌مرغ",
    "پشمچینی":"پشم‌چینی",
    "آتشسوزی":"آتش‌سوزی",
    "تصمیمگیری":"تصمیم‌گیری",
    "گرانقیمت":"گران‌قیمت",
    "ارزانقیمت":"ارزان‌قیمت",
    "رویهمرفته":"روی‌ هم‌ رفته",
    "معشوقاند":"معشوق اند",
    "لطفالله":"لطف‌‌الله",
    "آیتالله":"آیت‌الله",
    "مشخصشده":"مشخص‌شده",
    "نرمافزار":"نرم‌افزار",
    "سختافزار":"سخت‌افزار",
    "راهانداز":"راه‌انداز",
    "سیستمعامل":"سیستم عامل",
    "خانوادهاش":"خانواده‌اش",
    "تاریخگذاری":"تاریخ‌گذاری",
    "زغالسنگ":"زغال‌سنگ",
    "ذغالسنگ":"ذغال‌سنگ",
    "فارغالتحصیل":"فارغ‌التحصیل",
    "قریبالوقوع":"قریب‌الوقوع",
    "رأسالخیمه":"رأس‌الخیمه",
    "بیاحترامی":"بی‌احترامی",
    "اطلاعرسانی":"اطلاع‌رسانی",
    "عقبافتادگی":"عقب‌افتادگی",
    "کوخنشین":"کوخ‌نشین",
    "کاخنشین":"کاخ‌نشین",
    "بتهدار":"بته‌دار",
    "لکهدار":"لکه‌دار",
    "خندهدار":"خنده‌دار",
    "ریشسفید":"ریش‌سفید",
    "انگلیسیتبار":"انگلیسی‌تبار",
    "توصیفناپذیر":"توصیف‌ناپذیر",
    "تلقتلق":"تلق‌تلق",
    "گیاهشناسی":"گیاه‌شناسی",
    "جنینشناسی":"جنین‌شناسی",
    "زیستشناسی":"زیست‌شناسی",
    "روانشناسی":"روان‌شناسی",
    "گواهی‌نامهای":"گواهی‌نامه‌ای",
    "بیهسته":"بی‌هسته",
    "موبهموی":"مو‌به‌موی",
    "لیسانسیهی":"لیسانسه‌ی",
    "بیغذا":"بی‌غذا",
    "بیتردید":"بی‌تردید",
    "بیشک":"بی‌شک",
    "بیشکل":"بی‌شکل",
    "بیزحمت":"بی‌زحمت",
    "بیاعتنایی":"بی‌اعتنایی",
    "ماتمزده":"ماتم‌زده",
    "آدمربایان":"آدم‌ربایان",
    "زمستانخوابی":"زمستان‌خوابی",
    "دستاندازی":"دست‌اندازی",
    "نمیتوان":"نمی‌توان",
    "کتککاری":"کتک‌کاری",
    "حزبانتظار":"حزب‌انتظار",
    "عقبده":"عقیده",
    "نکردهای":"نکرده‌ای",
    "نمیآورم":"نمی‌آورم",
    "میآورند":"می‌آورند",
    "میپرورند":"می‌پرورند",
    "میپسندی":"می‌پسندی",
    "میپسندید":"می‌پسندید",
    "اصلن":"اصلا",
    "واقعن":"واقعا",
    "فعلن":"فعلا",
    "قبلن":"قبلا",
    "اولن":"اولا",
    "ثانین":"ثانیا",
    "ثالثن":"ثالثا",
    "حتمن":"حتما",
    "ابدن":"ابدا",
    "خواهشن":"خواهشا",
    "خاهشن":"خواهشا",
    "رسمن":"رسما",
    "قانونن":"قانونا",
    "شرعن":"شرعا",
    # "":"",
    "عایقبندی":"عایق‌بندی",
    "بمبگذاری":"بمب‌گذاری",
    "رقابتگرانه":"رقابت‌گرانه",
    "بیعلاقگی":"بی‌علاقگی",
    "خدمترسانی":"خدمت‌رسانی",
    "همپاشیده":"هم پاشیده",
    "همگونسازی":"همگون‌سازی",
    "ناهمگونسازی":"ناهمگون‌سازی",
    "میارزد":"می‌ارزد",
    "بیپیرایه":"بی‌پیرایه",
    "لغزشناپذیر":"لغزش‌ناپذیر",
    "بیاندازه":"بی‌اندازه",
    "قانونگزاران":"قانون‌گذاران",
    "شکلگیری":"شکل‌گیری",
    "آییننامه":"آیین‌نامه",
    "اسباببازی":"اسباب‌بازی",
    "بیعاری":"بی‌عاری",
    "بیآبرو":"بی‌آبرو",
    "کماهمیتتر":"کم‌اهمیت‌تر",
    "بیسوراخ":"بی‌سوراخ",
    "سهدوره":"سه دوره",
    "نومید‌کنندهای":"نومید‌کننده‌ای",
    "بچهدار":"بچه‌دار",
    "بیپردگی":"بی‌پردگی",
    "بیمیلی":"بی‌میلی",
    "خانهبهدوش":"خانه‌به‌دوش",
    "خانهبهدوشی":"خانه‌به‌دوشی",
    "دغلکاری":"دغل‌کاری",
    "پیریزی":"پی‌ریزی",
    "تختجمشید":"تخت‌جمشید",
    "مشتزنی":"مشت‌زنی",
    "بیوصیت‌نامه":"بی وصیت‌نامه",
    "کشتیگیر":"کشتی‌گیر",
    "ناراحت‌کنندهای":"ناراحت‌کننده‌ای",
    "خارقالعاده":"خارق‌العاده",
    "فوقالعاده":"فوق‌العاده",
    "خسیسترین":"خسیس‌ترین",
    "معروفترین":"معروف‌ترین",
    "بدینوسیله":"بدین وسیله",
    "بیتقصیر":"بی‌تقصیر",
    "راهحلهای":"راه‌حل‌های",
    "راهحل":"راه‌حل",
    "انعطافپذیری":"انعطاف‌پذیری",
    "بیعرضگی":"بی‌عرضگی",
    "بیعرضه":"بی‌عرضه",
    "خوشخوراک":"خوش‌خوراک",
    "معنیدار":"معنی‌دار",
    "بیخطا":"بی‌خطا",
    "ایستادهایم":"ایستاده‌ایم",
    "پیشبینی":"پیش‌بینی",
    "چاخانترین":"چاخان‌ترین",
    "هممحلیهای":"هم‌محلی‌های",
    "عیبدار":"عیب‌دار",
    "زبدهسالاری":"زبده‌سالاری",
    "صورتحساب":"صورت‌ حساب",
    "بیریخت":"بی‌ریخت",
    "بیادبی":"بی‌ادبی",
    "سهلالحصول":"سهل‌الحصول",
    "نگهداشتن":"نگه‌ داشتن",
    "کشتنشده":"کشت نشده",
    "مناعتنایی":"من اعتنایی",
    "گرفتهایم":"گرفته‌ایم",
    "تومانبرای":"تومان برای",
    "شهوتپرستی":"شهوت‌پرستی",
    "آزادیبخش":"آزادی‌بخش",
    "بیآبرویی":"بی‌آبرویی",
    "پیامرسان":"پیام‌رسان",
    "پیامرسانی":"پیام‌رسانی",
    "آهچه":"آه چه",
    "سایهروشن":"سایه روشن",
    "سستترین":"سست‌ترین",
    "شیرینزبانی":"شیرین‌زبانی",
    "فرانسویکه":"فرانسوی که",
    "موآخذه":"مواخذه",
    "دایرهالمعارف":"دایره‌المعارف",  
}


# 3.7) ی + suffixes
PATTERN_Y_SUFFIX = re.compile(r"\b([آ-ی]+ی)(ها|ام|ات|اش|ایم|اید|اند|است|مان|تان|شان|خواه|خواهی|خواهان|خواهانی)\b") 

# 3.8)
PATTERN_ALEF_ATTACH = re.compile(r"(?<![‌\s])([آ-یءٔ])(آ[آ-یءٔ]+)")

# =========================
# 4) Helpers
# =========================
def protect_exceptions(text: str):
    protected = {}
    for i, word in enumerate(sorted(EXCEPTIONS, key=len, reverse=True)):
        token = f"__EXC_{i}__"
        pattern1 = rf"\b{re.escape(word)}\b"

        if re.search(pattern1, text):
            text = re.sub(pattern1, token, text)
            protected[token] = word

    return text, protected


def restore_exceptions(text: str, protected: dict):
    for token, word in protected.items():
        text = text.replace(token, word)
    return text


def fix_alef_attachment(text: str) -> str:
    if not isinstance(text, str):
        return text
    text, protected = protect_exceptions(text)
    text = PATTERN_ALEF_ATTACH.sub(r"\1‌\2", text)
    text = restore_exceptions(text, protected)
    return text


def split_whitelist(whitelist: dict, chunk_size: int = 50):
    items = list(whitelist.items())
    for i in range(0, len(items), chunk_size):
        chunk = dict(items[i:i + chunk_size])
        pattern = re.compile(
            r'\b(' + '|'.join(re.escape(key) for key in chunk.keys()) + r')\b'
        )
        yield pattern, chunk

def apply_whitelist_chunked(text: str) -> str:
    if not isinstance(text, str):
        return text
    
    for pattern, chunk in split_whitelist(WHITELIST):
        text = pattern.sub(lambda m: chunk.get(m.group(0), m.group(0)), text)
    
    return text

# =========================
# 5) Main normalization
# =========================
def fix_text(text: str) -> str:
    if not isinstance(text, str):
        return text

    text = basic_unicode_normalize(text)
    text, protected = protect_exceptions(text)

    # 1. WHITELIST
    text = apply_whitelist_chunked(text)

    # 2. جدا کردن کلمات با "آ" از کلمه قبلی
    text=fix_alef_attachment(text)

    # 3. اعمال نیم‌فاصله برای کلمات جمع و ضمایر 'های/ها' (مثل فعالیتهای -> فعالیت‌های)
    text = PATTERN_GENERAL_HAY.sub(lambda m: m.group(1) + "‌" + m.group(2), text)

    # 4. کلمات مرکب (پردهبرداری -> پرده‌برداری)
    text = PATTERN_COMPOUND.sub(lambda m: m.group(1) + "‌" + m.group(2), text)

    text = PATTERN_ALEF_COMPOUND.sub(lambda m: m.group(1) + "‌" + m.group(2), text)

    # 5. ه + ضمایر (بودهام -> بوده‌ام)
    text = PATTERN_HE_OTHER.sub(lambda m: m.group(1) + "‌" + m.group(2), text)

    # 6. ه + اضافه (خانهی -> خانه‌ی)
    text = PATTERN_HE_EZAFE.sub(lambda m: m.group(1) + "‌" + "ی", text)

    # 7. واژه‌های ثابت
    text = PATTERN_HAMEH.sub("همه‌", text)
    text = PATTERN_FI_AL.sub("فی‌ال", text)
    text = PATTERN_BEYN_AL.sub("بین‌ال", text)

    # 8. به ...
    for tail in sorted(BE_ZWNJ_FORMS, key=len, reverse=True):
        text = re.sub(rf"\bبه{re.escape(tail)}\b", rf"به‌{tail}", text)
    for tail in sorted(BE_SPACE_FORMS, key=len, reverse=True):
        text = re.sub(rf"\bبه{re.escape(tail)}\b", rf"به {tail}", text)

    # 9. چه ...
    for tail in sorted(CHE_SPACE_FORMS, key=len, reverse=True):
        text = re.sub(rf"\bچه{re.escape(tail)}\b", rf"چه {tail}", text)

    # 10. ی + suffixes
    text = PATTERN_Y_SUFFIX.sub(lambda m: m.group(1) + "‌" + m.group(2), text)

    # 11. WHITELIST
    for src, tgt in WHITELIST.items():
        text = text.replace(src, tgt)
    text = apply_whitelist_chunked(text)


    # 12. بازگردانی استثناها
    text = restore_exceptions(text, protected)

    # 13. پاکسازی فاصله‌های مازاد و ZWNJ‌های تکراری
    text = re.sub(r"[ ]+\u200c[ ]+", "\u200c", text)
    text = re.sub(r"[ ]+\u200c", "\u200c", text)
    text = re.sub(r"\u200c[ ]+", "\u200c", text)
    text = re.sub(r"\u200c{2,}", "\u200c", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text



# =========================
# 6) Batch processing
# =========================
def normalize_dataframe(df: pd.DataFrame, columns=("ground_truth", "whisper", "mms")) -> pd.DataFrame:
    df = df.copy()
    for col in columns:
        if col in df.columns:
            df[col] = df[col].apply(fix_text)
    return df


def process_csv(input_path: str, output_path: str):
    df = pd.read_csv(input_path)
    df = normalize_dataframe(df)
    df.to_csv(output_path, index=False, encoding="utf-8-sig")


# =========================
# 7) Example usage
# =========================
if __name__ == "__main__":
    base_dir = Path("./raw data")
    files = [
        ("asr_cleaned_train.csv", "asr_cleaned_train_norm.csv"),
        ("asr_cleaned_validation.csv", "asr_cleaned_validation_norm.csv"),
        ("asr_cleaned_test.csv", "asr_cleaned_test_norm.csv"),
    ]

    for inp, out in files:
        in_path = base_dir / inp
        out_path = base_dir / out
        if in_path.exists():
            process_csv(str(in_path), str(out_path))
            print(f"Saved: {out_path}")
        else: 
            print(f"Missing: {in_path}")
